# Step 3 evidence: reproduce every number quoted during data checks

Each section below reproduces a finding from the Step 3 discussion, from the **raw** files (read-only).
Run top to bottom. If a number differs from the expected one in the check line, tell Claude before relying on it.

## E0. Raw-file integrity (run this first, and any time you suspect a file was touched)

In [8]:
import os
import pandas as pd

RAW = os.path.join('..', 'data-pack', 'data')
EXPECTED = dict(sku_master=(36,'sku_code'), stockist_master=(156,'stockist_code'), batch_master=(663,'batch_no'),
                primary_sales=(93809,'invoice_no'), expiry_claims=(4526,'claim_no'),
                cnf_stock_snapshot=(5015,'snapshot_date'), claim_emails=(30,'email_id'))

def load(n): return pd.read_csv(os.path.join(RAW, n + '.csv'), dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
R = {n: load(n) for n in EXPECTED}

for n, (rows, first_col) in EXPECTED.items():
    with open(os.path.join(RAW, n + '.csv'), encoding='utf-8') as fh: header = fh.readline().strip()
    ok_header = header.split(',')[0] == first_col          # header must be line 1 (a sort can push it to the bottom)
    ok_rows = len(R[n]) == rows
    print(f'{n:22s} header_first={ok_header}  rows={len(R[n])} (expected {rows})  {"OK" if ok_header and ok_rows else "<<< CHECK THIS FILE"}')
# primary_sales invoice INV/E01/2404/000889 must have a BLANK scheme_code in the original
ps0 = R['primary_sales']
print('INV/E01/2404/000889 scheme_code =', repr(ps0.loc[ps0.invoice_no == 'INV/E01/2404/000889', 'scheme_code'].iloc[0]), '(original is empty)')

sku_master             header_first=True  rows=36 (expected 36)  OK
stockist_master        header_first=True  rows=156 (expected 156)  OK
batch_master           header_first=True  rows=663 (expected 663)  OK
primary_sales          header_first=True  rows=93809 (expected 93809)  OK
expiry_claims          header_first=True  rows=4526 (expected 4526)  OK
cnf_stock_snapshot     header_first=True  rows=5015 (expected 5015)  OK
claim_emails           header_first=True  rows=30 (expected 30)  OK
INV/E01/2404/000889 scheme_code = '' (original is empty)


**Check:** every line says `OK`, and the last line shows `''` (empty). If not, restore the file from git (`git restore data-pack/data/<file>.csv`) before doing anything else.

## E1. Blank `original_invoice_ref` (expiry_claims): what it means

In [9]:
ec = R['expiry_claims']
blank = ec.original_invoice_ref == ''
print('blank invoice ref:', int(blank.sum()), 'of', len(ec), f'({blank.mean()*100:.1f}%)')
print('\nby status:\n', ec[blank].status.value_counts().to_string())
print('\nremarks on these rows:\n', ec[blank].remarks.replace('', '(no remark)').value_counts().to_string())
print('\nBlank-ref rows by claim_type:\n', ec[blank].claim_type.value_counts().to_string())
print('\nColumns that could hold Area Manager approval:', [c for c in ec.columns if 'approv' in c.lower() or 'manager' in c.lower()] or 'NONE')

blank invoice ref: 884 of 4526 (19.5%)

by status:
 status
Credited    802
Pending      64
Rejected     18

remarks on these rows:
 remarks
Party says invoice misplaced                              286
Invoice copy not attached, accepted on stockist ledger    258
Inv ref missing, verified on batch                        258
(no remark)                                                64
Duplicate of earlier claim                                 18

Blank-ref rows by claim_type:
 claim_type
Near expiry            510
Expired                298
Breakage in transit     48
Short supply            28

Columns that could hold Area Manager approval: NONE


**Check:** 884 blank (19.5%). Credited 802, Pending 64, Rejected 18. Remarks: 286 / 258 / 258 / 18 / 64 with none. No approval column exists, so R2's second half is untestable in the data.

## E2. Blank `credit_note_no`: it follows the status

In [10]:
cn_blank = ec.credit_note_no == ''
print(pd.crosstab(ec.status, cn_blank.map({True: 'credit note blank', False: 'has credit note'})))
print('\nRejected claims, remarks:', ec[ec.status == 'Rejected'].remarks.value_counts().to_dict())

credit_note_no  credit note blank  has credit note
status                                            
Credited                        0             4129
Pending                       366                0
Rejected                       31                0

Rejected claims, remarks: {'Duplicate of earlier claim': 31}


**Check:** blank = Pending 366 + Rejected 31 = 397. All 4,129 Credited have a number. All 31 Rejected say `Duplicate of earlier claim`.

## E3. Blank `ordered_qty`: documented, it is a later row of a split order line (`01` p3 section 5.4)

In [11]:
ps = R['primary_sales'].copy()
num = lambda s: pd.to_numeric(s.str.replace(',', '', regex=False), errors='coerce')
for c in ['ordered_qty','invoiced_qty','free_qty','pts_inr','net_value_inr']: ps[c] = num(ps[c])
# same normalisation and exact-duplicate rule as the cleaning notebook (746 rows); psd = de-duplicated copy, ps stays the raw rows for E4
psd = ps.copy()
psd['stockist_code'] = psd.stockist_code.str.strip().str.upper()
psd['scheme_code'] = psd.scheme_code.replace({'': 'NO_SCHEME', 'NA': 'NO_SCHEME', '-': 'NO_SCHEME'})
psd = psd.drop_duplicates()
print('rows after dropping exact duplicates:', len(psd), '(expect 93,063)')

ob = psd.ordered_qty.isna()
cnt = psd.groupby(['invoice_no', 'sku_code']).ordered_qty.transform('count')
print('blank ordered_qty rows:', int(ob.sum()), f'({ob.mean()*100:.1f}%)')
print('  ... each has ANOTHER row in the same invoice+SKU that carries the ordered_qty:', int((cnt[ob] >= 1).sum()), 'of', int(ob.sum()))
print('  first row of any order line with a blank ordered_qty:', int(psd.groupby(['invoice_no', 'sku_code']).head(1).ordered_qty.isna().sum()))

# short supply must be measured per ORDER LINE (invoice_no + sku_code): sum invoiced over the batch rows vs the ordered quantity
ol = psd.groupby(['invoice_no', 'sku_code']).agg(ordered=('ordered_qty', 'max'), invoiced=('invoiced_qty', 'sum')).reset_index()
print('\norder lines:', len(ol))
print('  filled in full         :', int((ol.invoiced == ol.ordered).sum()), f'({(ol.invoiced == ol.ordered).mean()*100:.1f}%)')
print('  short (partly filled)  :', int(((ol.invoiced > 0) & (ol.invoiced < ol.ordered)).sum()), f'({((ol.invoiced > 0) & (ol.invoiced < ol.ordered)).mean()*100:.1f}%)')
print('  not supplied (0)       :', int((ol.invoiced == 0).sum()), f'({(ol.invoiced == 0).mean()*100:.1f}%)')
print('  invoiced above ordered :', int((ol.invoiced > ol.ordered).sum()))
print(f'  packs invoiced / packs ordered = {ol.invoiced.sum() / ol.ordered.sum() * 100:.2f}%   (indicative: adds packs of different SKUs; QBR fill rate is reconciled in 3.6)')
print('\nROW-level comparison (the wrong way): rows with ordered > invoiced =', int((psd[~ob].ordered_qty > psd[~ob].invoiced_qty).sum()), ' <- inflated by split lines')

rows after dropping exact duplicates: 93063 (expect 93,063)
blank ordered_qty rows: 3043 (3.3%)
  ... each has ANOTHER row in the same invoice+SKU that carries the ordered_qty: 3043 of 3043
  first row of any order line with a blank ordered_qty: 0

order lines: 87186
  filled in full         : 74133 (85.0%)
  short (partly filled)  : 10425 (12.0%)
  not supplied (0)       : 2628 (3.0%)
  invoiced above ordered : 0
  packs invoiced / packs ordered = 91.97%   (indicative: adds packs of different SKUs; QBR fill rate is reconciled in 3.6)

ROW-level comparison (the wrong way): rows with ordered > invoiced = 21038  <- inflated by split lines


**Check:** 3,043 blank rows, **all** 3,043 sit on a split line whose first row carries the order; 0 first rows are blank. Order lines: 87,186 = **74,133** full + **10,425** short + **2,628** not supplied; invoiced above ordered: 0. The row-level figure (21,038) is wrong because a line split over two batches shows the full order on the first row and only part of it invoiced. **Short supply is measured per order line.**

## E4. What `scheme_code` blank / `NA` / `-` mean (inference: all three = no scheme)

In [12]:
ps['grp'] = ps.scheme_code.where(ps.scheme_code.isin(['NA', '-', '']), 'real code').replace({'': '(blank)'})
print(ps.groupby('grp').agg(rows=('free_qty', 'size'), with_free_goods=('free_qty', lambda x: int((x > 0).sum()))))
print('\ntop real codes:', ps[ps.grp == 'real code'].scheme_code.value_counts().head(5).to_dict())

            rows  with_free_goods
grp                              
(blank)    48446                0
-          16294                0
NA         16347                0
real code  12722            11023

top real codes: {'QS-FY25Q3-A': 915, 'QS-FY25Q2-A': 911, 'QS-FY25Q4-A': 909, 'QS-FY26Q1-A': 909, 'QS-FY26Q2-A': 894}


**Check:** (blank) 48,446, `-` 16,294, `NA` 16,347 with **zero** free-goods rows each; real codes 12,722 with 11,023 free-goods rows. This supports, but does not prove, that the three mean no scheme. Confirm with the client.

## E5. Stock snapshot: conflicting quantities, row counts per C&F, stock-out markers (blank batch + zero quantity = SKU out of stock, `01` p3 section 5.5)

In [13]:
sn = R['cnf_stock_snapshot'].copy()
sn['cnf'] = sn.cnf_code.str.replace(r'^CF(?=[A-Z]\d)', 'CF-', regex=True)
sn['qty'] = num(sn.qty_on_hand_packs)
print('rows per C&F (after fixing CFE01 style):\n', sn.cnf.value_counts().to_string())
hb = sn[sn.batch_no != '']
k = ['snapshot_date', 'cnf', 'sku_code', 'batch_no']
g = hb[hb.duplicated(k, keep=False)]
print('\nrows in conflicting groups:', len(g), ' groups:', g.groupby(k).ngroups)
print('groups where qty differs:', int((g.groupby(k).qty.nunique() > 1).sum()), ' where expiry differs:', int((g.groupby(k).expiry_date.nunique() > 1).sum()))
print('conflicting groups by C&F:\n', g.drop_duplicates(k).cnf.value_counts().to_string())
so = sn[(sn.batch_no == '') & (sn.qty == 0)]
print('\nblank-batch, qty 0 rows:', len(so))

rows per C&F (after fixing CFE01 style):
 cnf
CF-E01    1406
CF-S01     791
CF-S02     768
CF-W01     764
CF-N01     683
CF-N02     603

rows in conflicting groups: 1683  groups: 831
groups where qty differs: 831  where expiry differs: 0
conflicting groups by C&F:
 cnf
CF-E01    308
CF-S01    134
CF-S02    129
CF-W01    127
CF-N01     80
CF-N02     53

blank-batch, qty 0 rows: 270


**Check:** 831 conflicting groups (1,683 rows), quantity differs in all 831, expiry in none. After the C&F code fix, CF-E01 has 1,406 rows against 603-791 for the others. Conflicting groups appear at **all six** C&Fs (E01 308, S01 134, S02 129, W01 127, N01 80, N02 53), so this is not one site's habit, though Kolkata is over-represented. Cause unknown: client question. 270 blank-batch rows, all quantity 0.

## E6. Emails that chase credit notes (evidence for claim-processing delays)

In [14]:
em = R['claim_emails']
hit = em[(em.subject + ' ' + em.body).str.contains('credit note|resend', case=False)]
for _, r in hit.iterrows():
    print(r.email_id, r.received_date, r.from_name, r.stockist_code_by_mailbox_rule, '|', r.subject, '|', r.body.replace('\r\n', ' ')[:200], '\n')

EML-022 01-10-2026 New Medicos STK0043 | Resending claim | Sir, resending our claim, please check: Amox-Clav 625, batch D2500228, exp Sep 2026, 241 strips, inv INV/N02/2506/039882
Regards 

EML-019 05-10-2026 Matrix Medicos STK0143 | Resending expiry claim | Dear sir,
As discussed on phone, resending our claim for Ambroxol syp batch A2530386, 206 strips. Credit note not received yet.
Dhanyavaad 

EML-026 09-10-2026 City Medicos STK0093 | credit note? | Sir what happened to our B-complex caps claim, batch H2410154? No claim number with us.
Thanks & regards
Billing 

EML-005 12-10-2026 Ganesh Pharma Distributors STK0051 | expiry maal wapas | Namaste ji,
Humare paas Methylcobalamin 1500 ka 147 strip expiry ke paas hai, batch E2510396. Exp Oct 2026.
Kripya credit note bhej dijiye, party payment hold kar rahi hai.
Ganesh Pharma Distributors 

EML-021 16-10-2026 Sai Pharma Traders STK0102 | claim again | Resending claim, Diclo gel G2420385, 79 strips (3 more found).
Thanks & regards
Billing 

**Check:** six emails print: EML-019 (Matrix Medicos, credit note not received yet), EML-005 (Ganesh Pharma Distributors, wants credit note, payment on hold), EML-026 (City Medicos, asks what happened to its claim, no claim number), EML-001 (Royal Pharma Traders, asks for a credit note), and resends EML-022 and EML-021. Always cite email IDs like these, never from memory.